# Use examples of [edges](https://github.com/romainsacchi/edges)

Author: [romainsacchi](https://github.com/romainsacchi)

This notebook shows examples on how to use `edge` to use location-specific
characterization factors in the characterization matrix of `bw2calc`.

## Requirements

* **Pyhton 3.10 or higher (up to 3.11) is highly recommended**
* `bw2calc >= 2.0.0`

# Use case with [brightway2](https://brightway.dev/)

`brightway2` is an open source LCA framework for Python.
To use `premise` from `brightway2`, it requires that you have an activated `brightway2` project with a `biosphere3` database as well as an [ecoinvent](https://ecoinvent.prg) v.3 cut-off or consequential database registered in that project. Please refer to the brightway [documentation](https://brightway.dev) if you do not know how to create a project and install ecoinvent.

In [ ]:
from edges import EdgeLCIA, get_available_methods
import bw2data

### List of available methods


In [ ]:
list(get_available_methods())

### Biosphere edges-based LCIA

The `EdgeLCIA` class is a subclass of `bw2calc.LCA` that allows for the use of location-specific characterization factors in the characterization matrix of `bw2calc`. `SpatialLCA`can therefore be used as a drop-in replacement for `bw2calc.LCA` in most cases.

In [ ]:
bw2data.projects

In [ ]:
# activate the bw project
bw2data.projects.set_current("ecoinvent-3.10-cutoff")

In [ ]:
act = bw2data.Database("ecoinvent-3.10.1-cutoff").random()
act

In [ ]:
method = ('AWARE 2.0', 'Country', 'all', 'yearly')

In [ ]:
LCA = EdgeLCIA({act: 1}, method)
LCA.lci()
LCA.apply_strategies()
LCA.evaluate_cfs()
LCA.lcia()
LCA.score

In [ ]:
# let's check locations that have been ignored, if any
LCA.ignored_locations

### Generate dataframe of characterization factors used

The `generate_cf_table` method generates a dataframe of the characterization factors used in the calculation. One can see the characterization factors used for each exchange in the system. For deterministic regionalized runs, `split_aggregate_consumers=True` expands weighted fallback consumer regions such as RER, GLO, RoW, or RoE into country rows.

In [ ]:
df = LCA.generate_cf_table(split_aggregate_consumers=True)

In [ ]:
# we can see under the "CF" column
# the characterization factors used for each exchange in the system
df

In [ ]:
df.groupby("consumer location")[["consumer location", "impact"]].sum().sort_values(ascending=False, by="impact").head(20).plot(kind="bar")

In [ ]:
df.groupby("consumer location")[["consumer location", "impact"]].sum()

In [ ]:
df_map = (
    df.groupby("consumer location", as_index=True)["impact"]
    .sum()
    .to_frame()
)

print(df_map.head())


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pycountry
import numpy as np

# 1. Aggregate impacts by country
df_map = (
    df.groupby("consumer location", as_index=True)["impact"]
    .sum()
    .to_frame()
)

# 2. Build clean mapping table from index
data = pd.DataFrame({
    "iso2": df_map.index.astype(str),
    "impact": df_map["impact"].values,
})

# 3. Convert ISO-2 to ISO-3
def iso2_to_iso3(code):
    try:
        c = pycountry.countries.get(alpha_2=code.strip())
        return c.alpha_3 if c else None
    except Exception:
        return None

data["iso3"] = data["iso2"].apply(iso2_to_iso3)

# 4. Remove non-country regions like WEU, WECC, etc.
data = data.dropna(subset=["iso3"]).copy()

# Optional: log scale
data["impact_plot"] = np.where(data["impact"] > 0, np.log10(data["impact"]), np.nan)

# 5. Load country polygons
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)

# 6. Merge
world = world.merge(
    data[["iso3", "impact_plot"]],
    how="left",
    left_on="ISO_A3",
    right_on="iso3",
)

print("Matched countries:", world["impact_plot"].notna().sum())

# 7. Plot
fig, ax = plt.subplots(figsize=(16, 8))

world.plot(
    column="impact_plot",
    cmap="OrRd",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax,
)

ax.set_title("Impact by consumer location")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
df_map_ = (
    df_.groupby("consumer location", as_index=True)["impact"]
    .sum()
    .to_frame()
)

print(df_map_.head())


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pycountry
import numpy as np

# 1. Aggregate impacts by country
df_map = (
    df_.groupby("consumer location", as_index=True)["impact"]
    .sum()
    .to_frame()
)

# 2. Build clean mapping table from index
data = pd.DataFrame({
    "iso2": df_map.index.astype(str),
    "impact": df_map["impact"].values,
})

# 3. Convert ISO-2 to ISO-3
def iso2_to_iso3(code):
    try:
        c = pycountry.countries.get(alpha_2=code.strip())
        return c.alpha_3 if c else None
    except Exception:
        return None

data["iso3"] = data["iso2"].apply(iso2_to_iso3)

# 4. Remove non-country regions like WEU, WECC, etc.
data = data.dropna(subset=["iso3"]).copy()

# Optional: log scale
data["impact_plot"] = np.where(data["impact"] > 0, np.log10(data["impact"]), np.nan)

# 5. Load country polygons
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)

# 6. Merge
world = world.merge(
    data[["iso3", "impact_plot"]],
    how="left",
    left_on="ISO_A3",
    right_on="iso3",
)

print("Matched countries:", world["impact_plot"].notna().sum())

# 7. Plot
fig, ax = plt.subplots(figsize=(16, 8))

world.plot(
    column="impact_plot",
    cmap="OrRd",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax,
)

ax.set_title("Impact by consumer location")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
LCA.scenario_cfs[:5]

In [ ]:
print(data.head())
print(data["iso3"].dropna().unique()[:10])

print(world.columns)
print(world[["ISO_A3"]].head())

print(world["impact_plot"].notna().sum())

In [ ]:
df_ = LCA.generate_cf_table(split_aggregate_consumers=True)

In [ ]:
df_.groupby("consumer location")[["consumer location", "impact"]].sum().sort_values(ascending=False, by="impact").head(20).plot(kind="bar")

In [ ]:
len(df_)

In [ ]:
locs = {
    "Auvergne-Rhône-Alpes": "FR-ARA",
    "Nouvelle-Aquitaine": "FR-NA",
    "Occitanie": "FR-OC",
    "Pays de la Loire": "FR-PL",
    "Provence-Alpes-Côte d'Azur": "FR-PACA",
    "Bourgogne-Franche-Comté": "FR-BFC",
    "Bretagne": "FR-BR",
    "Centre-Val de Loire": "FR-CVL",
    "Corse": "FR-CO",
    "Grand Est": "FR-GE",
    "Hauts-de-France": "FR-HF",
    "Île-de-France": "FR-IL",
    "Normandie": "FR-NO"
}

In [ ]:
if "h2" in bw2data.databases:
    del bw2data.databases["h2"]

In [ ]:
lci = {
    ("h2", f"hydrogen production, from electrolysis {loc}"): {
        "name": "hydrogen production, from electrolysis",
        "reference product": "hydrogen",
        "location": loc,
        "unit": "kilogram",
        "classifications": [
            ("ISIC", "Hydrogen production"),
        ],
        "exchanges": [
            {
                "name": "hydrogen production, from electrolysis",
                "reference product": "hydrogen",
                "unit": "kilogram",
                "amount": 1,
                "location": loc,
                "type": "production",
                "input": ("h2", f"hydrogen production, from electrolysis {loc}"),
            },
            {
                "name": "electricity production, wind, 1-3MW turbine, offshore",
                "reference product": "electricity, high voltage",
                "unit": "kilowatt hour",
                "amount": 54,
                "location": "FR",
                "type": "technosphere",
                "input": ('ecoinvent-3.10-cutoff', '7ba2c49140d98d7c941cef85f6f58d75'),
            },
            {
                "name": "electrolyzer production, 1MWe, PEM, Stack",
                "reference product": "electrolyzer, 1MWe, PEM, Stack",
                "unit": "unit",
                "amount": 1.34989E-06,
                "location": "RER",
                "type": "technosphere",
                "input": ('h2_pem', '7b498745a482ad10549d1d4330c7abf0')
            },
            {
                "name": "electrolyzer production, 1MWe, PEM, Balance of Plant",
                "reference product": "electrolyzer, 1MWe, PEM, Balance of Plant",
                "unit": "unit",
                "amount": 3.37473E-07,
                "location": "RER",
                "type": "technosphere",
                "input": ('h2_pem', 'f3fda92c6f40b652aee44ac8e68053a4')
            },
            {
                "name": "treatment of fuel cell stack, 1MWe, PEM",
                "reference product": "used fuel cell stack, 1MWe, PEM",
                "unit": "unit",
                "amount": -1.34989E-06,
                "location": "RER",
                "type": "technosphere",
                "input": ('h2_pem', 'affe261eea51b2ddb28189f84c7c28b4')
            },
            {
                "name": "treatment of fuel cell balance of plant, 1MWe, PEM",
                "reference product": "used fuel cell balance of plant, 1MWe, PEM",
                "unit": "unit",
                "amount": -3.37473E-07,
                "location": "RER",
                "type": "technosphere",
                "input": ('h2_pem', '937fc767b9547738e849fe73fee8c51f')
            },
            {
                "name": "Water, well, in ground",
                "categories": ("natural resource", "water"),
                "unit": "cubic meter",
                "amount": 12 / 1000, # 12l per kg of hydrogen
                "type": "biosphere",
                "input": ('biosphere', '67c40aae-d403-464d-9649-c12695e43ad8')
            },
            {
                # needed for the GLO AWARE to work, because
                # it only characterizes evaporation of water
                "name": "Water",
                "categories": ("air",),
                "unit": "cubic meter",
                "amount": 12 / 1000, # 12l per kg of hydrogen
                "type": "biosphere",
                "input": ('biosphere', '075e433b-4be4-448e-9510-9a5029c1ce94')
            }

        ],
    } for loc in locs.values()
}

In [ ]:
bw2data.Database("h2").write(lci)

In [ ]:
from edges.analysis import compare_activities_by_grouped_leaves
import bw2data
bw2data.projects.set_current("bw25_ei310")

In [ ]:
method = ('AWARE 1.2c', 'Country', 'non', 'irri', 'yearly')
FU = [act for act in bw2data.Database("h2")]
df = compare_activities_by_grouped_leaves(
    FU,
    method,
    max_level=1,
    output_format="pandas",
    mode="absolute",
    cutoff=1e-2,
)

In [ ]:
df.columns = [
    'activity',
    'product',
    'location',
    'unit',
    'total',
    'impacts directs',
    'production électrique',
    'fabrication électrolyseur'
]

In [ ]:
# méthode globale
globalmethod = ('ecoinvent-3.10',
  'EF v3.1 EN15804',
  'water use',
  'user deprivation potential (deprivation-weighted water consumption)')

In [ ]:
from bw2analyzer.comparisons import compare_activities_by_grouped_leaves as ac

FU = [act for act in bw2data.Database("h2")]
df_glo = ac(
    FU[:1],
    globalmethod,
    max_level=1,
    output_format="pandas",
    mode="absolute",
    cutoff=1e-2,
)

In [ ]:
df_glo.columns = [
    'activity',
    'product',
    'location',
    'unit',
    'total',
    'impacts directs',
    'production électrique',
    'fabrication électrolyseur'
]

In [ ]:
df_glo.loc[:, "location"] = "GLO"

In [ ]:
# append the global method to the dataframe
import pandas as pd
df = pd.concat([df, df_glo])

In [ ]:
import matplotlib.pyplot as plt
# Prepare the data for the stacked bar chart
locations = df["location"]
values = df.loc[:, "impacts directs":]

# Plot the stacked bar chart
fig, ax = plt.subplots(figsize=(8, 4))
values.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")

# Set chart details
ax.set_xticks(range(len(locations)))
ax.set_xticklabels(locations, rotation=0)
ax.set_xlabel("Région")
ax.set_ylabel("$m^3$-éq./kg $H_2$")
ax.set_title("Score d'impact AWARE pour la production d'hydrogène par électrolyse en France")
plt.legend(title="Sources d'impact")
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
df

In [ ]:
FU[0]

In [ ]:
LCA = EdgeLCIA({FU[0]: 1}, method)
LCA.lci()
LCA.lcia()
LCA.score

In [ ]:
df_cf = LCA.generate_cf_table(split_aggregate_consumers=True)

In [ ]:
df_cf

In [ ]:
df_cf.to_excel("cf_h2.xlsx")

In [ ]:
import matplotlib.pyplot as plt

# Grouping the data by consumer location and consumer name to sum the impact
grouped_data = df_cf.groupby(['consumer location', 'consumer name'])['impact'].sum()
grouped_data



In [ ]:
# remove row that contribute less than 1% to the total impact
grouped_data = grouped_data[grouped_data > 0.02 * grouped_data.sum()]

In [ ]:
# make a stacked bar chart to show results per consumer location on teh x-axis and consumer name on the legend
fig, ax = plt.subplots(figsize=(10, 6))
grouped_data.unstack().plot(kind='bar', stacked=True, ax=ax)
# reformat labels on teh x.axis
#ax.set_xticklabels(grouped_data.index.levels[0], rotation=45)
# set y axis label
ax.set_ylabel('m$^3$-eq/kg H$_2$')

## Technosphere edges-based LCIA

The `EdgeLCIA` class can also be used to calculate LCIA results for technosphere exchanges.

In [ ]:
from edges import EdgeLCIA
import bw2data
bw2data.projects.set_current("bw25_ei310")

In [ ]:
act = [a for a in bw2data.Database("ecoinvent-3.10-cutoff") if "NMC" in a["name"]][0]
act

In [ ]:
method = ('RELICS', 'copper')
LCA = EdgeLCIA({act: 1}, method)
LCA.lci()
LCA.lcia()
LCA.score

### Using GeoPolRisk (under development)

In [ ]:
method = ('GeoPolRisk', '2024')
LCA = EdgeLCIA({act: 1}, method)
LCA.lci()
LCA.lcia()
LCA.score

In [ ]:
df = LCA.generate_cf_table(split_aggregate_consumers=True)
df